# Name That LLM: train the Mistral-7B-v0.1 LoRA member on Colab

Generated by `name_that_llm/remote_train.py colab`; do not edit by hand.

Trains the same 5-fold, two-stage LoRA member as the Kaggle training notebook, without using Kaggle GPU quota, then stages the output for upload as a private Kaggle dataset.

**Before running:**
1. **Runtime → Change runtime type → A100 GPU** (an L4 works, at about the speed of Kaggle's two T4s).
2. **Secrets** (key icon on the left): add `KAGGLE_API_TOKEN` (a token from kaggle.com/settings → API Tokens; or the legacy `KAGGLE_USERNAME` and `KAGGLE_KEY`) and give this notebook access. You must have joined the competition on Kaggle.
3. Run the timing run first (`MODE = 'timing'`, ~10 min), check its s/step, then set `MODE = 'full'` and run all again.

Nothing is written to Google Drive. The full run backs itself up to a private Kaggle dataset (stage 1 every 20 minutes, each fold as it finishes) and, when started again after a disconnect, restores that backup and continues from it. `scripts/colab/` starts and watches the same run without this tab.

In [ ]:
VARIANT = 'mistral7b-full'
MODE = 'timing'  # 'timing' (~10 min speed check) or 'full'

## Setup

In [ ]:
import os
import subprocess

from google.colab import userdata

print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'],
                     capture_output=True, text=True).stdout)
# a new-style API token (KGAT_...); else the legacy username + key pair
try:
    os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN').strip()
except userdata.SecretNotFoundError:
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')

In [ ]:
%cd /content
!test -d ntl || git clone https://github.com/kemalhanan/name-that-llm.git ntl
%cd /content/ntl
!git pull -q
!pip install -q uv
!uv sync -q --group lora

## Inputs

The competition data, the Open Pool and the base model, downloaded with kagglehub to this runtime's disk (about 15 GB for a 7-8B model).

In [ ]:
!uv run python -m name_that_llm.remote_train fetch {VARIANT}

## Train

The timing run writes to the runtime's disk and starts fresh each time. The full run restores the last backup from your private Kaggle dataset `lora-out-mistral7b-full` (if there is one), continues from it, uploads every new save, and uploads the finished member at the end. Rerun this cell after a disconnect; attach the dataset to the submission notebook when it is done.

In [ ]:
if MODE == 'full':
    !uv run python -m name_that_llm.remote_train unattended {VARIANT}
else:
    !rm -rf /content/timing_out
    !uv run python -m name_that_llm.remote_train run {VARIANT} --mode timing --out /content/timing_out